## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Online orders per day and a campaign start: X-Rs chart

**X-Rs (Individuals and Moving Range) Control Chart** · a worked example from the StatNex teaching guide

**The story.** An online shop records one number per working day: how many orders came in. A marketing campaign started on day 22.

**The question.** Does the chart show that something changed when the campaign started?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import IPython.display
import matplotlib

matplotlib.use("Agg")


def show_new_figures(*_, shown=set()):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path.name, path.stat().st_mtime_ns) not in shown:
            shown.add((path.name, path.stat().st_mtime_ns))
            IPython.display.display(IPython.display.Image(filename=str(path)))


events = IPython.get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `10_control_chart/x_rs_chart.csv` (26 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Date,Day number,Online orders
2025-09-01,1.0,389.0
2025-09-02,2.0,404.0
2025-09-03,3.0,395.0
2025-09-04,4.0,424.0
2025-09-05,5.0,428.0
2025-09-08,6.0,408.0
2025-09-09,7.0,447.0
2025-09-10,8.0,456.0
2025-09-11,9.0,416.0
2025-09-12,10.0,421.0
2025-09-15,11.0,428.0
2025-09-16,12.0,418.0
2025-09-17,13.0,406.0
2025-09-18,14.0,431.0
2025-09-19,15.0,444.0
2025-09-22,16.0,435.0
2025-09-23,17.0,460.0
2025-09-24,18.0,449.0
2025-09-25,19.0,455.0
2025-09-26,20.0,392.0
2025-09-29,21.0,414.0
2025-09-30,22.0,559.0
2025-10-01,23.0,493.0
2025-10-02,24.0,514.0
2025-10-03,25.0,517.0
2025-10-06,26.0,499.0
""")

## Helper: control_signals

A step that no library does, written out. In the code download it is the file `control_signals.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Reading a Shewhart control chart: the two pattern rules, and the chart itself.

SciPy and statsmodels have no control charts, so both are written out here.

A point outside the 3-sigma limits is the obvious signal. A chart is also read for its
shape, because two patterns inside the limits are about as unlikely as a point outside
when nothing has changed (Nelson's rules 2 and 3; ISO 7870-2 and JIS Z 9020-2 list the
same tests):

* 9 points in a row on one side of the centre line -- the level may have shifted;
* 6 rises in a row (7 points climbing), or 6 falls -- a trend.

These are the two StatNex applies. Each run is reported once, from its first point to
its last, however long it lasts: nine in a row sets the warning off, and a run of
eighteen is still one run. A point exactly on the centre line, or a missing one (the
first moving range), breaks a run.
"""

import numpy as np
from matplotlib.figure import Figure


def stretches(signs, minimum):
    """Each stretch of at least `minimum` equal, non-zero signs, as (first, last) positions."""
    found, start = [], 0
    for i in range(1, len(signs) + 1):
        if i == len(signs) or signs[i] != signs[start]:
            if signs[start] != 0 and i - start >= minimum:
                found.append((start, i - 1))
            start = i
    return found


def pattern_signals(label, values, centre):
    """Messages for the runs and trends in one chart's points (a pandas Series)."""
    numbers = list(values.index)
    points = values.to_numpy(float)
    found = []

    side = np.where(np.isfinite(points), np.sign(points - np.asarray(centre, float)), 0.0)
    for first, last in stretches(side, 9):
        where = "above" if side[first] > 0 else "below"
        found.append(f"{label}: points {numbers[first]} to {numbers[last]} are {last - first + 1} in a row "
                     f"{where} the centre line. The level of the process may have shifted.")

    # Step i joins points i - 1 and i; six steps one way are seven points climbing or falling.
    steps = np.zeros(len(points))
    for i in range(1, len(points)):
        if np.isfinite(points[i]) and np.isfinite(points[i - 1]):
            steps[i] = np.sign(points[i] - points[i - 1])
    for first, last in stretches(steps, 6):
        where = "rise" if steps[first] > 0 else "fall"
        found.append(f"{label}: points {numbers[first - 1]} to {numbers[last]} {where} "
                     f"monotonically. This is a sign of a change in trend.")
    return found


def draw(path, title, values, centre, upper, lower, xlabel="Subgroup"):
    """One chart: the points, the centre line, and the limits (stepped if they vary)."""
    fig = Figure(figsize=(7, 4.5))
    ax = fig.subplots()
    x = np.asarray(values.index)
    ax.plot(x, values.to_numpy(float), marker="o", color="black", linewidth=1)
    for line, style in ((upper, "--"), (centre, "-"), (lower, "--")):
        ax.step(x, np.broadcast_to(np.asarray(line, float), x.shape), where="mid",
                linestyle=style, color="grey", linewidth=1)
    out = (values > upper) | (values < lower)
    ax.plot(x[out.to_numpy()], values[out].to_numpy(float), "o", color="red")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    fig.savefig(path, dpi=150)

## Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
# draw and pattern_signals: defined in the helper cell control_signals above (control_signals.py in the code download)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
# One measurement per row, in time order. A moving range is the difference between
# neighbours, so a blank or text cell stops the chart instead of being skipped.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
column = "Online orders"
x = pd.to_numeric(data[column], errors="coerce")
if x.isna().any():
    raise ValueError("Every cell must hold a number: a gap breaks the moving ranges.")
x.index = np.arange(1, len(x) + 1)  # point numbers 1, 2, ...

## Control-chart constants

In [ ]:
# With one reading per period there is no subgroup to measure the everyday spread, so it
# comes from neighbours: the moving range |x(i) - x(i-1)| is the range of a "subgroup" of
# two. The process standard deviation is estimated as MR-bar / d2, with d2 = 1.128 for a
# range of two, and the limits are 3 of those either side:
#   X chart:  CL +/- E2 x MR-bar, where E2 = 3 / d2 = 2.66
#   Rs chart: UCL = D4 x MR-bar = 3.267 x MR-bar, LCL = 0
# These are the table's values, as StatNex uses them.
E2, D4 = 2.66, 3.267


def chart(points):
    """X and moving-range lines for these points; the first point has no moving range."""
    moving = points.diff().abs()
    x_bar, mr_bar = points.mean(), moving.mean()  # mean() skips the first point's blank
    table = pd.DataFrame({"X": points, "CL": x_bar, "UCL": x_bar + E2 * mr_bar,
                          "LCL": x_bar - E2 * mr_bar, "moving range": moving,
                          "Rs CL": mr_bar, "Rs UCL": D4 * mr_bar, "Rs LCL": 0.0})
    return table


def outside(table):
    return ((table["X"] > table["UCL"]) | (table["X"] < table["LCL"])
            | (table["moving range"] > table["Rs UCL"]) | (table["moving range"] < table["Rs LCL"]))


points = x

## Control chart data

In [ ]:
# One row per point: the reading against the X chart's centre line (CL) and limits (UCL,
# LCL), then its moving range against the Rs chart's. The first point has no moving
# range. A point outside its limits signals a change beyond everyday variation; a big
# moving range marks a sudden jump from the point before. The limits assume the
# readings are independent and roughly normal: a trend or a weekly cycle makes them
# misleading.
table = chart(points)
print()
print(table)
print(f"\nPoints outside the limits: {list(table.index[outside(table)])}")

## The charts

In [ ]:
# Besides points outside the limits, a chart is read for patterns that are just as
# unlikely by chance: 9 points in a row on one side of the centre line, or 6 rises (or
# falls) in a row.
for message in (pattern_signals("X chart", table["X"], table["CL"])
                + pattern_signals("Moving range", table["moving range"], table["Rs CL"])):
    print(message)
Path("figures").mkdir(exist_ok=True)
draw("figures/x_chart.png", "X chart", table["X"], table["CL"], table["UCL"], table["LCL"],
     xlabel="Point")
draw("figures/rs_chart.png", "Rs chart", table["moving range"], table["Rs CL"], table["Rs UCL"],
     table["Rs LCL"], xlabel="Point")

## What the result says

This chart follows one value per day, the number of orders. The centre line (CL) is the average, 442 orders. The control limits mark the reach of ordinary chance variation: from 376 (LCL) to 509 (UCL).

For 21 days every value is inside the limits. Then day 22, the day the campaign started, has 559 orders, and days 24 and 25 have 514 and 517. All three are above the UCL. The moving range is the change from one day to the next. On day 22 it is 145, far above its own UCL of 81.8, so the level jumped in one step.

So yes, something changed on day 22, and orders stayed higher. But the chart only dates the change. It cannot prove that the campaign caused it.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.